# WEEK - 7 ASSIGNMENT

In [0]:
# Step 1: Load Dataset
df = spark.read.csv("/Volumes/workspace/default/myfiles/Student.csv", header=True,inferSchema=True)

In [0]:
display(df.limit(20))

StudentID,Name,Department,Year,Gender,Age,Attendance,Marks,CGPA,City,Scholarship,Email
1001,Aarav Sharma,CS,1,M,18,92,85,8.8,Delhi,Yes,aarav.sharma@univ.edu
1002,Aditi Rao,ECE,2,F,19,88,78,8.2,Mumbai,No,aditi.rao@univ.edu
1003,Arjun Verma,ME,3,M,21,74,65,7.1,Bangalore,No,arjun.verma@univ.edu
1004,Ananya Iyer,CS,1,F,18,95,92,9.4,Chennai,Yes,ananya.iyer@univ.edu
1005,Aditya Patel,IT,4,M,22,68,58,6.5,Ahmedabad,No,aditya.patel@univ.edu
1006,Avani Singh,ECE,2,F,20,81,74,7.8,Lucknow,No,avani.singh@univ.edu
1007,Ayush Gupta,CS,3,M,21,89,88,8.9,Delhi,Yes,ayush.gupta@univ.edu
1008,Amrita Nair,EE,1,F,18,90,81,8.3,Kochi,No,amrita.nair@univ.edu
1009,Akash Das,ME,2,M,20,72,62,6.9,Kolkata,No,akash.das@univ.edu
1010,Anjali Mishra,IT,3,F,21,85,null,7.6,Noida,No,anjali.mishra@univ.edu


In [0]:
# Step 2: Remove duplicates and handle null values
from pyspark.sql.functions import avg, when, col
df1 = df.dropDuplicates()
df1 = df1.fillna("Unknown")
#Converting string values to int
df1 = df1.withColumn(
    "Marks",
    when(col("Marks") == "null", None).otherwise(col("Marks").cast("int"))
)
avg_marks = df1.select(avg("Marks")).first()[0]
avg_cgpa = df1.select(avg("CGPA")).first()[0]
df1 = df1.fillna({"Marks": int(avg_marks), "CGPA": avg_cgpa})
display(df1)

StudentID,Name,Department,Year,Gender,Age,Attendance,Marks,CGPA,City,Scholarship,Email
1007,Ayush Gupta,CS,3,M,21,89,88,8.9,Delhi,Yes,ayush.gupta@univ.edu
1060,Zoya Akhtar,EE,4,F,22,86,79,8.1,Mumbai,No,zoya.akhtar@univ.edu
1062,Bipasha Basu,ME,3,F,21,67,56,6.1,Kolkata,No,bipasha.basu@univ.edu
1073,Manoj Bajpayee,ECE,2,M,20,78,70,7.3,Patna,No,manoj.bajpayee@univ.edu
1095,Lucky Ali,ME,1,M,19,66,54,5.9,Bangalore,No,lucky.ali@univ.edu
1008,Amrita Nair,EE,1,F,18,90,81,8.3,Kochi,No,amrita.nair@univ.edu
1037,Pranav Mistry,ECE,3,M,21,80,72,7.4,Surat,No,pranav.mistry@univ.edu
1039,Pankaj Tripathi,EE,2,M,20,71,61,6.8,Patna,No,pankaj.tripathi@univ.edu
1043,Rohan Mehra,ECE,4,M,22,76,68,7.1,Gurgaon,No,rohan.mehra@univ.edu
1051,Tejaswi Prakash,CS,4,F,22,96,96,9.7,Mumbai,Yes,tejaswi.prakash@univ.edu


root
 |-- StudentID: integer (nullable = true)
 |-- Name: string (nullable = false)
 |-- Department: string (nullable = false)
 |-- Year: integer (nullable = true)
 |-- Gender: string (nullable = false)
 |-- Age: integer (nullable = true)
 |-- Attendance: string (nullable = false)
 |-- Marks: string (nullable = false)
 |-- CGPA: double (nullable = true)
 |-- City: string (nullable = false)
 |-- Scholarship: string (nullable = false)
 |-- Email: string (nullable = false)



In [0]:
# Step 3 : Creating second dataset for incremental data 
from pyspark.sql import Row

new = [
    Row(
        StudentID=1003,Name="Arjun Verma",Department="CS",Year=3,Gender="M",Age=21,Attendance="90",
        Marks=91,CGPA=9.1,City="Bangalore",Scholarship="Yes",Email="arjun.verma@univ.edu"
    ),

    Row(
        StudentID=1021,Name="Rahul Kumar",Department="IT",Year=2,Gender="M",Age=20,Attendance="88",
        Marks=84,CGPA=8.4,City="Hyderabad",Scholarship="No",Email="rahul.kumar@univ.edu"
    ),

    Row(
        StudentID=1022, Name="Sneha Reddy", Department="ECE", Year=4,Gender="F",Age=22,
        Attendance="95",Marks=93,CGPA=9.4,City="Chennai",Scholarship="Yes",Email="sneha.reddy@univ.edu"
    )
]

new_df = spark.createDataFrame(new)

display(new_df)
new_df.write.format("delta").mode("overwrite").saveAsTable("student_incremental")

StudentID,Name,Department,Year,Gender,Age,Attendance,Marks,CGPA,City,Scholarship,Email
1003,Arjun Verma,CS,3,M,21,90,91,9.1,Bangalore,Yes,arjun.verma@univ.edu
1021,Rahul Kumar,IT,2,M,20,88,84,8.4,Hyderabad,No,rahul.kumar@univ.edu
1022,Sneha Reddy,ECE,4,F,22,95,93,9.4,Chennai,Yes,sneha.reddy@univ.edu


In [0]:
%sql
-- step 4 : Apply Merge Operation
MERGE INTO student_data AS t
USING student_incremental AS s
ON t.StudentID = s.StudentID

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,3,0,0


In [0]:
# Step 5 : Validation
print("Total Rows:", df.count())
final_df = spark.table("student_data")
print("Total Rows:", final_df.count())
# as data got merged so no. of rows remained same 

Total Rows: 100
Total Rows: 100


In [0]:
# step 6 : Final Dataset
display(final_df.limit(20))

StudentID,Name,Department,Year,Gender,Age,Attendance,Marks,CGPA,City,Scholarship,Email
1007,Ayush Gupta,CS,3,M,21,89,88,8.9,Delhi,Yes,ayush.gupta@univ.edu
1060,Zoya Akhtar,EE,4,F,22,86,79,8.1,Mumbai,No,zoya.akhtar@univ.edu
1062,Bipasha Basu,ME,3,F,21,67,56,6.1,Kolkata,No,bipasha.basu@univ.edu
1073,Manoj Bajpayee,ECE,2,M,20,78,70,7.3,Patna,No,manoj.bajpayee@univ.edu
1095,Lucky Ali,ME,1,M,19,66,54,5.9,Bangalore,No,lucky.ali@univ.edu
1008,Amrita Nair,EE,1,F,18,90,81,8.3,Kochi,No,amrita.nair@univ.edu
1037,Pranav Mistry,ECE,3,M,21,80,72,7.4,Surat,No,pranav.mistry@univ.edu
1039,Pankaj Tripathi,EE,2,M,20,71,61,6.8,Patna,No,pankaj.tripathi@univ.edu
1043,Rohan Mehra,ECE,4,M,22,76,68,7.1,Gurgaon,No,rohan.mehra@univ.edu
1051,Tejaswi Prakash,CS,4,F,22,96,96,9.7,Mumbai,Yes,tejaswi.prakash@univ.edu


# Summary
The Student dataset was loaded into a Delta table and cleaned by removing duplicate records and handling missing values. A second dataset was created to simulate incremental data. Using the MERGE operation, existing records were updated. Finally, the results were validated by checking the row count, verifying duplicate records, and displaying the final dataset.